# 🚗 Driver Guardian AI — SpatioTemporal ConvGRU Training Pipeline (Kaggle Edition)
## Huấn luyện Mô hình Phát hiện Ngủ gật Trực tiếp từ Video Thô qua NMSFreeDetector PAFPN & ConvGRU

> **Đặc tả Kỹ thuật:**
> - **Kiến trúc mô hình**: `ConvGRUClassifier` (Bảo toàn 2D Spatial Map $40\times 40$ qua trục thời gian với cơ chế Chú ý Kép Dual Attention).
> - **Bộ trích xuất đặc trưng**: `ChunkedBackboneNeckExtractor` (Cắt nhỏ mini-chunk trên GPU, trích xuất đa tỷ lệ $p_3, p_4, p_5$ từ trọng số PAFPN NMSFreeDetector).
> - **Dữ liệu**: Nạp video thô (`.mp4`, `.avi`, `.mkv`), lấy mẫu đều đặn (`sample_interval=0.2s`), tensor CPU uint8 siêu nhẹ.
> - **Tối ưu hóa**: AdamW, Cosine Annealing LR, Mixed Precision (AMP FP16), Gradient Accumulation, Early Stopping.
> - **Cơ chế Khôi phục (Resume)**: Hỗ trợ tiếp tục train từ bất kỳ epoch nào (`epoch_N.pt`), bí danh (`'last'`, `'best'`) hoặc file tùy chọn.
> - **Môi trường**: Kaggle Notebook GPU Kernel (Tesla T4 / P100 16GB VRAM, Ubuntu Linux).

In [ ]:
# 1. Cài đặt các thư viện bổ sung nếu cần và kiểm tra dependencies
import os
import sys
import time
import math
import random
import csv
import json
import shutil
import zipfile
import subprocess
import contextlib
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Any, Optional, Union

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

# Thiết lập hiển thị đồ thị inline mượt mà
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print(f"[✓] PyTorch Version : {torch.__version__}")
print(f"[✓] OpenCV Version  : {cv2.__version__}")
print(f"[✓] CUDA Sẵn sàng   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[✓] Thiết bị GPU    : {torch.cuda.get_device_name(0)}")
    print(f"[✓] Số lượng GPU    : {torch.cuda.device_count()}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[✓] Tổng VRAM       : {vram_gb:.2f} GB")
else:
    print("[!] CẢNH BÁO: Không phát hiện GPU. Vui lòng bật GPU Accelerator (T4 x1 hoặc P100) trong menu Kaggle Notebook!")


In [ ]:
# 2. Cố định Seed ngẫu nhiên để đảm bảo tính tái lập (Reproducibility)
def seed_everything(seed: int = 42) -> None:
    """Cố định seed cho tất cả các thư viện để kết quả huấn luyện có thể tái lập hoàn toàn."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"[✓] Đã cố định toàn bộ hạt giống ngẫu nhiên: SEED = {seed}")

seed_everything(42)

## ⚙️ Section 2: Cấu hình Đường dẫn & Siêu tham số Huấn luyện
Toàn bộ siêu tham số được đồng bộ 100% từ cấu hình chuẩn `configs/config.yaml`.
Trên môi trường Kaggle:
- Thư mục đầu vào dữ liệu chỉ đọc: `/kaggle/input/...`
- Thư mục xuất checkpoint, nhật ký và đồ thị có quyền ghi: `/kaggle/working/...`
- Hỗ trợ đầy đủ các tham số **Resume Training** (`resume`, `resume_epoch`, `save_epoch_interval`, `max_keep_ckpts`).

In [ ]:
# Tự động nhận diện môi trường làm việc Kaggle vs Local
IS_KAGGLE = os.path.exists("/kaggle")

def auto_detect_dataset_dir() -> str:
    """Tự động phát hiện thư mục dataset trong /kaggle/input hoặc cục bộ."""
    if not IS_KAGGLE:
        return "E:/LSTM/data_processed"
    candidates = [
        "/kaggle/input/driver-guardian-dataset/data_processed",
        "/kaggle/input/driver-guardian-dataset",
        "/kaggle/input/data-processed",
        "/kaggle/input/dataset",
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    # Quét tự động trong /kaggle/input nếu có thư mục chứa dataset_merged_split.csv hoặc train/
    if os.path.exists("/kaggle/input"):
        for root, dirs, files in os.walk("/kaggle/input"):
            if "dataset_merged_split.csv" in files or "train" in dirs:
                return root
    return "/kaggle/input/driver-guardian-dataset/data_processed"

def auto_detect_backbone_ckpt() -> str:
    """Tự động phát hiện trọng số NMSFreeDetector best.pt trong /kaggle/input."""
    if not IS_KAGGLE:
        return "checkpoints/best.pt"
    candidates = [
        "/kaggle/input/driver-guardian-backbone/best.pt",
        "/kaggle/input/driver-guardian-backbone/landmark_train/best.pt",
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    if os.path.exists("/kaggle/input"):
        for root, dirs, files in os.walk("/kaggle/input"):
            if "best.pt" in files:
                return os.path.join(root, "best.pt")
    return "/kaggle/input/driver-guardian-backbone/best.pt"

@dataclass
class KaggleTrainConfig:
    # ---------------- 1. ĐƯỜNG DẪN DỮ LIỆU & CHECKPOINT ----------------
    # Tự động phát hiện đường dẫn thực tế trên Kaggle
    dataset_dir: str = auto_detect_dataset_dir()
    manifest_file: str = "dataset_merged_split.csv"
    backbone_neck_checkpoint: str = auto_detect_backbone_ckpt()
    
    output_dir: str = "/kaggle/working/runs" if IS_KAGGLE else "logs/runs"
    checkpoint_dir: str = "/kaggle/working/checkpoints" if IS_KAGGLE else "checkpoints/experiments"
    tb_log_dir: str = "/kaggle/working/logs/tensorboard" if IS_KAGGLE else "logs/tensorboard"
    history_csv_path: str = "/kaggle/working/training_history.csv" if IS_KAGGLE else "logs/training_history.csv"
    experiment_name: str = "deepgru_raw_nmsfree"

    # ---------------- 2. THAM SỐ LẤY MẪU VIDEO ----------------
    sample_interval: float = 0.2     # Lấy mẫu 1 frame mỗi 0.2s (~5 FPS)
    seq_len: Optional[int] = None    # None: nạp toàn bộ clip và padding động
    image_size: Tuple[int, int] = (640, 640)
    video_exts: Tuple[str, ...] = (".avi", ".mp4", ".mkv", ".mov")
    min_frames: int = 10
    use_augmentation: bool = True

    # ---------------- 3. THAM SỐ DATALOADER ----------------
    batch_size: int = 8
    val_batch_size: int = 4
    num_workers: int = 2             # Đặt 2 để tối ưu tốc độ và không tràn /dev/shm (2GB) trên Kaggle
    val_num_workers: int = 0
    pin_memory: bool = True if torch.cuda.is_available() else False
    shuffle: bool = True
    chunk_size: int = 32             # Cắt nhỏ batch khi forward GPU qua Backbone tránh OOM

    # ---------------- 4. KIẾN TRÚC MÔ HÌNH CONVGRU ----------------
    cnn_neck_channels: List[int] = field(default_factory=lambda: [64, 128, 256])
    cnn_out_channels: int = 448
    cnn_spatial_size: Tuple[int, int] = (20, 20)
    adapter_dropout: float = 0.25
    input_dim: int = 128
    hidden_dim: int = 128
    num_layers: int = 2
    num_classes: int = 2
    dropout: float = 0.35
    supervision_mode: str = "attention_pooling"

    # ---------------- 5. HÀM MẤT MÁT, TỐI ƯU & SCHEDULER ----------------
    loss_type: str = "ce"            # "ce" cho multi-class (2 classes), "bce" cho binary 1-logit
    pos_weight: Optional[float] = None
    bce_reduction: str = "mean"
    epochs: int = 30
    lr0: float = 0.001
    lr_min_factor: float = 0.01      # lr_min = 1e-5
    weight_decay: float = 0.0001
    optimizer: str = "adamw"
    betas: Tuple[float, float] = (0.9, 0.999)
    momentum: float = 0.9
    grad_clip_norm: float = 1.0
    gradient_accumulation_steps: int = 4
    empty_cache_interval: int = 1
    use_scheduler: bool = True

    # ---------------- 6. EARLY STOPPING & GIÁM SÁT TENSORBOARD ----------------
    early_stopping: bool = True
    patience: int = 10
    monitor_metric: str = "val_f1"
    monitor_mode: str = "max"
    min_delta: float = 0.0001
    val_interval_epochs: int = 1

    # Giám sát TensorBoard theo từng Step (Per-Step Metrics)
    enable_step_logging: bool = True          # Bật/tắt tính toán và ghi nhận per-step metrics lên TensorBoard
    log_step_interval: int = 5                # Số batch steps giữa các lần ghi TensorBoard (mặc định = 5 steps)
    flush_step_interval: int = 50             # Số batch steps giữa các lần flush bộ đệm xuống đĩa (mặc định = 50 steps)

    # ---------------- 7. TIẾP TỤC HUẤN LUYỆN (RESUME) & CHECKPOINT ----------------
    enable_resume: bool = False               # True: tự động resume khi khởi tạo Trainer
    resume: Optional[Union[str, int]] = None  # File path, số epoch (VD: 5), hoặc bí danh ('last', 'best')
    resume_epoch: Optional[int] = None        # Tùy chọn ép buộc số thứ tự epoch bắt đầu
    save_epoch_interval: int = 1             # Lưu checkpoint epoch_{N}.pt mỗi N epoch
    save_best_only: bool = False             # Nếu True: chỉ lưu best.pt và last.pt để tiết kiệm đĩa
    max_keep_ckpts: int = 5                  # Tự động giữ lại N checkpoint gần nhất tránh tràn 20GB đĩa Kaggle

    # ---------------- 8. PHẦN CỨNG ----------------
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    amp: bool = True

# Khởi tạo đối tượng cấu hình
cfg = KaggleTrainConfig()

print(f"[*] Môi trường thực thi : {'KAGGLE NOTEBOOK' if IS_KAGGLE else 'LOCAL ENVIRONMENT'}")
print(f"[*] Thư mục Dataset     : {cfg.dataset_dir} (Tồn tại: {Path(cfg.dataset_dir).exists()})")
print(f"[*] Trọng số Backbone   : {cfg.backbone_neck_checkpoint} (Tồn tại: {Path(cfg.backbone_neck_checkpoint).exists()})")
print(f"[*] Thư mục Checkpoints : {cfg.checkpoint_dir}")
print(f"[*] Thư mục Logs TB     : {cfg.tb_log_dir} (Step logging: {cfg.enable_step_logging}, interval: {cfg.log_step_interval} steps)")
print(f"[*] Lịch sử CSV         : {cfg.history_csv_path}")
print(f"[*] Cấu hình Resume     : enable_resume={cfg.enable_resume}, resume={cfg.resume}")


## 🧠 Section 3: Định nghĩa Bộ Trích xuất Đặc trưng Backbone PAFPN
Định nghĩa đầy đủ các khối CNN và mạng `BackboneNeck` (PAFPN từ NMSFreeDetector) cùng lớp trích xuất `ChunkedBackboneNeckExtractor`.
Module này chạy hoàn toàn độc lập mà **không cần import từ bất kỳ tệp ngoài nào**.

In [ ]:
# 1. Các khối tích chập chuẩn hóa của NMSFreeDetector PAFPN
def autopad(k, p=None, d=1):
    """Tự động tính toán padding để giữ nguyên kích thước không gian."""
    if p is not None:
        return p
    return (d * (k - 1) + 1) // 2

class Conv(nn.Module):
    def __init__(self, c1, c2, k=1, s=1, p=None, g=1, d=1, act=True):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, k, s, autopad(k, p, d), groups=g, dilation=d, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

class Bottleneck(nn.Module):
    def __init__(self, c1, c2, shortcut=True, e=0.5):
        super().__init__()
        c_ = int(c2 * e)
        self.cv1 = Conv(c1, c_, 3, 1)
        self.cv2 = Conv(c_, c2, 3, 1)
        self.add = shortcut and c1 == c2

    def forward(self, x):
        y = self.cv2(self.cv1(x))
        return x + y if self.add else y

class C2f(nn.Module):
    def __init__(self, c1, c2, n=1, shortcut=True, e=0.5):
        super().__init__()
        self.c = int(c2 * e)
        self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.cv2 = Conv((2 + n) * self.c, c2, 1, 1)
        self.m = nn.ModuleList(Bottleneck(self.c, self.c, shortcut, e=1.0) for _ in range(n))

    def forward(self, x):
        y = list(self.cv1(x).chunk(2, 1))
        for m in self.m:
            y.append(m(y[-1]))
        return self.cv2(torch.cat(y, 1))

class CIB(nn.Module):
    def __init__(self, c1, c2, shortcut=True, e=0.5, large_kernel=False):
        super().__init__()
        c_ = int(c2 * e)
        k = 7 if large_kernel else 3
        self.block = nn.Sequential(
            Conv(c1, c1, k, 1, g=c1),
            Conv(c1, 2 * c_, 1, 1),
            Conv(2 * c_, 2 * c_, k, 1, g=2 * c_),
            Conv(2 * c_, c2, 1, 1),
            Conv(c2, c2, k, 1, g=c2),
        )
        self.add = shortcut and c1 == c2

    def forward(self, x):
        y = self.block(x)
        return x + y if self.add else y

class C2fCIB(C2f):
    def __init__(self, c1, c2, n=1, shortcut=False, e=0.5, large_kernel=False):
        super().__init__(c1, c2, n, shortcut, e)
        self.m = nn.ModuleList(
            CIB(self.c, self.c, shortcut, e=1.0, large_kernel=large_kernel) for _ in range(n)
        )

class SPPF(nn.Module):
    def __init__(self, c1, c2, k=5):
        super().__init__()
        c_ = c1 // 2
        self.cv1 = Conv(c1, c_, 1, 1)
        self.cv2 = Conv(4 * c_, c2, 1, 1)
        self.m = nn.MaxPool2d(k, stride=1, padding=k // 2)

    def forward(self, x):
        x = self.cv1(x)
        y1 = self.m(x)
        y2 = self.m(y1)
        y3 = self.m(y2)
        return self.cv2(torch.cat((x, y1, y2, y3), 1))

class SCDown(nn.Module):
    def __init__(self, c1, c2, k=3, s=1, p=None, d=1, act=True):
        super().__init__()
        self.cv1 = Conv(c1, c2, 1, 1, act=act)
        self.cv2 = Conv(c2, c2, k, s, p=p, g=c2, d=d, act=False)

    def forward(self, x):
        return self.cv2(self.cv1(x))

class Attention(nn.Module):
    def __init__(self, dim, num_heads=4, mlp_ratio=2.0, layer_scale=1e-2):
        super().__init__()
        hidden_dim = int(dim * mlp_ratio)
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv = nn.Conv2d(dim, 3 * dim, 1, bias=False)
        self.proj = Conv(dim, dim, 1, 1, act=False)
        self.pe = Conv(dim, dim, 3, 1, g=dim, act=False)
        self.ffn = nn.Sequential(
            Conv(dim, hidden_dim, 1, 1),
            Conv(hidden_dim, dim, 1, 1, act=False),
        )
        self.gamma1 = nn.Parameter(layer_scale * torch.ones(dim))
        self.gamma2 = nn.Parameter(layer_scale * torch.ones(dim))

    def forward(self, x):
        B, C, H, W = x.shape
        qkv = self.qkv(x).reshape(B, 3, self.num_heads, self.head_dim, H * W)
        q, k, v = qkv.permute(1, 0, 2, 4, 3).contiguous().unbind(0)
        v_spatial = v.transpose(-2, -1).reshape(B, C, H, W)
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0)
        out = out.transpose(-2, -1).reshape(B, C, H, W)
        out = self.proj(out + self.pe(v_spatial))
        x = x + self.gamma1.to(out.dtype).view(1, -1, 1, 1) * out
        out = self.ffn(x)
        return x + self.gamma2.to(out.dtype).view(1, -1, 1, 1) * out

class C2fPSA(nn.Module):
    def __init__(self, c1, c2, n=1, e=0.5, num_heads=4):
        super().__init__()
        self.c = int(c2 * e)
        self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.cv2 = Conv(2 * self.c, c2, 1, 1)
        self.m = nn.Sequential(*(Attention(self.c, num_heads=num_heads) for _ in range(n)))

    def forward(self, x):
        a, b = self.cv1(x).chunk(2, 1)
        return self.cv2(torch.cat((a, self.m(b)), 1))

# 2. Kiến trúc Backbone và PAFPN
class Backbone(nn.Module):
    def __init__(self, w=(64, 128, 256, 512, 1024), n=(3, 6, 6, 3)):
        super().__init__()
        c0, c1, c2, c3, c4 = w
        n0, n1, n2, n3 = n

        self.stem = Conv(3, c0, 3, 2)
        self.stage1 = nn.Sequential(
            Conv(c0, c1, 3, 2),
            C2f(c1, c1, n=n0, shortcut=True),
        )
        self.stage2 = nn.Sequential(
            Conv(c1, c2, 3, 2),
            C2f(c2, c2, n=n1, shortcut=True),
        )
        self.stage3 = nn.Sequential(
            SCDown(c2, c3, 3, 2),
            C2fCIB(c3, c3, n=n2, shortcut=True),
        )
        self.stage4 = nn.Sequential(
            SCDown(c3, c4, 3, 2),
            C2fCIB(c4, c4, n=n3, shortcut=True, large_kernel=True),
            SPPF(c4, c4, k=5),
            C2fPSA(c4, c4, n=2, e=0.5),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        p3 = self.stage2(x)
        p4 = self.stage3(p3)
        p5 = self.stage4(p4)
        return p3, p4, p5

class PAFPN(nn.Module):
    def __init__(self, chs=(256, 512, 1024), n=3):
        super().__init__()
        c3, c4, c5 = chs
        self.up = nn.Upsample(scale_factor=2, mode="nearest")

        # Top-down
        self.c2f_p4 = C2f(c5 + c4, c4, n=n, shortcut=True)
        self.c2f_p3 = C2f(c4 + c3, c3, n=n, shortcut=False)

        # Bottom-up
        self.down3 = Conv(c3, c3, 3, 2)
        self.c2f_n4 = C2fCIB(c3 + c4, c4, n=n, shortcut=True)
        self.down4 = SCDown(c4, c4, 3, 2)
        self.c2f_n5 = C2fCIB(c4 + c5, c5, n=n, shortcut=True, large_kernel=True)

    def forward(self, p3, p4, p5):
        # Top-down
        x = self.up(p5)
        x = torch.cat([x, p4], 1)
        p4_td = self.c2f_p4(x)

        x = self.up(p4_td)
        x = torch.cat([x, p3], 1)
        p3_out = self.c2f_p3(x)

        # Bottom-up
        x = self.down3(p3_out)
        x = torch.cat([x, p4_td], 1)
        p4_out = self.c2f_n4(x)

        x = self.down4(p4_out)
        x = torch.cat([x, p5], 1)
        p5_out = self.c2f_n5(x)

        return p3_out, p4_out, p5_out

class BackboneNeck(nn.Module):
    def __init__(self, backbone: nn.Module, neck: nn.Module):
        super().__init__()
        self.backbone = backbone
        self.neck = neck

    def forward(self, x):
        return self.neck(*self.backbone(x))

In [ ]:
# 3. Lớp trích xuất đặc trưng Mini-Chunk trên GPU (ChunkedBackboneNeckExtractor)
class ChunkedBackboneNeckExtractor(nn.Module):
    """
    Trích xuất đặc trưng đa tầng (p3, p4, p5) trực tiếp trên GPU.
    Chia nhỏ video thành các chunk (chunk_size=32) để bảo đảm không vượt trần VRAM.
    """
    def __init__(
        self,
        checkpoint_path: Union[str, Path],
        device: str = "cuda",
        chunk_size: int = 32,
        use_fp16: bool = True,
        img_size: int = 640
    ) -> None:
        super().__init__()
        self.checkpoint_path = Path(checkpoint_path)
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.chunk_size = max(1, chunk_size)
        self.use_fp16 = use_fp16
        self.img_size = img_size
        self.model = self._load_model()

    def _load_model(self) -> nn.Module:
        # Cấu hình chuẩn của NMSFreeDetector PAFPN Backbone:
        # trunk_backbone_w = (16, 32, 64, 128, 256) -> c3=64, c4=128, c5=256
        # trunk_backbone_n = (1, 2, 2, 1), trunk_neck_n = 1
        backbone_w = (16, 32, 64, 128, 256)
        backbone_n = (1, 2, 2, 1)
        neck_n = 1

        checkpoint = None
        if self.checkpoint_path.exists():
            print(f"[*] Nạp checkpoint BackboneNeck từ: {self.checkpoint_path}")
            try:
                checkpoint = torch.load(self.checkpoint_path, map_location="cpu", weights_only=False)
            except Exception as e:
                print(f"[!] Lỗi khi nạp checkpoint Backbone: {e}")

        # Tự động đồng bộ từ metadata của checkpoint nếu có
        if checkpoint and isinstance(checkpoint, dict) and "metadata" in checkpoint:
            meta_arch = checkpoint["metadata"].get("architecture", {})
            if "trunk_backbone_w" in meta_arch:
                backbone_w = tuple(meta_arch["trunk_backbone_w"])
            if "trunk_backbone_n" in meta_arch:
                backbone_n = tuple(meta_arch["trunk_backbone_n"])
            if "trunk_neck_n" in meta_arch:
                neck_n = int(meta_arch["trunk_neck_n"])
            print(f"[✓] Đồng bộ kiến trúc từ metadata checkpoint: w={backbone_w}, n={backbone_n}, neck_n={neck_n}")

        c3, c4, c5 = backbone_w[2], backbone_w[3], backbone_w[4]
        backbone = Backbone(w=backbone_w, n=backbone_n)
        neck = PAFPN(chs=(c3, c4, c5), n=neck_n)
        model = BackboneNeck(backbone, neck).eval()

        if checkpoint is not None:
            state_dict = checkpoint.get("ema") or checkpoint.get("model") or checkpoint
            
            filtered_sd = {}
            for k, v in state_dict.items():
                if k.startswith("backbone.") or k.startswith("neck."):
                    filtered_sd[k] = v
                elif not k.startswith("head."):
                    filtered_sd[k] = v
            
            msg = model.load_state_dict(filtered_sd, strict=False)
            print(f"[✓] Đã nạp trọng số Backbone thành công! (Matched keys: {len(filtered_sd)})")
        else:
            print(f"[!] CẢNH BÁO: Không tìm thấy checkpoint '{self.checkpoint_path}'. Khởi tạo ngẫu nhiên để thử nghiệm.")

        for param in model.parameters():
            param.requires_grad = False

        return model.to(self.device)

    def forward(self, batch_frames: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        B, T, C, H, W = batch_frames.shape
        flat_frames = batch_frames.view(B * T, C, H, W)

        p3_list, p4_list, p5_list = [], [], []

        with torch.inference_mode():
            for i in range(0, B * T, self.chunk_size):
                chunk = flat_frames[i : i + self.chunk_size].to(self.device, non_blocking=True)
                if chunk.dtype == torch.uint8:
                    chunk = chunk.float() / 255.0

                if self.use_fp16 and self.device.type == "cuda":
                    with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                        out3, out4, out5 = self.model(chunk)
                    # Giữ nguyên FP16 giúp tiết kiệm 50% dung lượng VRAM lưu trữ feature cache
                    p3_list.append(out3.half())
                    p4_list.append(out4.half())
                    p5_list.append(out5.half())
                else:
                    out3, out4, out5 = self.model(chunk)
                    p3_list.append(out3.float())
                    p4_list.append(out4.float())
                    p5_list.append(out5.float())

        p3_flat = torch.cat(p3_list, dim=0)
        p4_flat = torch.cat(p4_list, dim=0)
        p5_flat = torch.cat(p5_list, dim=0)

        p3 = p3_flat.view(B, T, *p3_flat.shape[1:])
        p4 = p4_flat.view(B, T, *p4_flat.shape[1:])
        p5 = p5_flat.view(B, T, *p5_flat.shape[1:])

        return p3, p4, p5


## 🤖 Section 4: Kiến trúc Mô hình Không gian - Thời gian `ConvGRUClassifier`
Tích hợp:
1. **`SpatialReductionNeck`**: Cổ giảm kênh đa tỷ lệ ($448 \rightarrow 64$ kênh tại lưới $40 \times 40$).
2. **`ConvGRUCell` & `ConvGRU`**: Gộp cổng Kernel Fusion ($2\times H$), bảo toàn đặc trưng 2D và hỗ trợ Sequence Masking.
3. **Cơ chế Chú ý Kép Dual Attention**:
   - `SpatialAttentionPooling`: Tập trung vào vùng mặt, mắt, miệng trên lưới không gian.
   - `TemporalAttentionPooling`: Gom tụ chuỗi thời gian, loại bỏ các frame zero-padding qua `seq_lens`.
4. **Phân loại**: MLP Head với Dropout.

In [ ]:
# 1. Tế bào và Tầng Convolutional GRU (ConvGRUCell & ConvGRU)
class ConvGRUCell(nn.Module):
    """Tế bào ConvGRU đơn tầng với cơ chế Kernel Fusion (gộp cổng Reset và Update)."""
    def __init__(self, input_dim: int, hidden_dim: int, kernel_size: int = 3):
        super().__init__()
        self.input_dim = int(input_dim)
        self.hidden_dim = int(hidden_dim)
        padding = kernel_size // 2

        self.conv_gates = nn.Conv2d(
            in_channels=self.input_dim + self.hidden_dim,
            out_channels=2 * self.hidden_dim,
            kernel_size=kernel_size,
            padding=padding,
            bias=True
        )
        self.conv_candidate = nn.Conv2d(
            in_channels=self.input_dim + self.hidden_dim,
            out_channels=self.hidden_dim,
            kernel_size=kernel_size,
            padding=padding,
            bias=True
        )
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for m in (self.conv_gates, self.conv_candidate):
            nn.init.orthogonal_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(
        self,
        x_t: torch.Tensor,
        h_prev: torch.Tensor,
        mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        combined = torch.cat([x_t, h_prev], dim=1)
        gates = self.conv_gates(combined)
        r_gate, z_gate = torch.chunk(gates, 2, dim=1)
        r = torch.sigmoid(r_gate)
        z = torch.sigmoid(z_gate)

        combined_candidate = torch.cat([x_t, r * h_prev], dim=1)
        h_tilde = torch.tanh(self.conv_candidate(combined_candidate))
        h_next = (1.0 - z) * h_prev + z * h_tilde

        if mask is not None:
            h_next = torch.where(mask, h_next, h_prev)

        return h_next


class ConvGRU(nn.Module):
    """Module ConvGRU đa tầng hỗ trợ Dynamic Spatial Resolution và Masking."""
    def __init__(self, input_dim: int, hidden_dim: int, num_layers: int = 2, kernel_size: int = 3, dropout: float = 0.2):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.dropout_p = dropout

        cell_list = []
        for i in range(num_layers):
            cur_in = input_dim if i == 0 else hidden_dim
            cell_list.append(ConvGRUCell(cur_in, hidden_dim, kernel_size=kernel_size))
        self.cells = nn.ModuleList(cell_list)
        self.dropout = nn.Dropout2d(p=dropout) if dropout > 0 else nn.Identity()

    def forward(
        self,
        x: torch.Tensor,
        seq_lens: Optional[torch.Tensor] = None,
        h_0: Optional[List[torch.Tensor]] = None
    ) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        B, T, C, H, W = x.shape
        device, dtype = x.device, x.dtype

        if h_0 is None:
            h_states = [torch.zeros(B, self.hidden_dim, H, W, device=device, dtype=dtype) for _ in range(self.num_layers)]
        else:
            h_states = list(h_0)

        time_masks = None
        if seq_lens is not None:
            t_steps = torch.arange(T, device=device).unsqueeze(0)
            time_masks = (t_steps < seq_lens.unsqueeze(1)).unsqueeze(2).unsqueeze(3).unsqueeze(4)

        cur_layer_input = x
        for layer_idx, cell in enumerate(self.cells):
            h_cur = h_states[layer_idx]
            output_steps = []

            for t in range(T):
                x_t = cur_layer_input[:, t]
                mask_t = time_masks[:, t] if time_masks is not None else None
                h_cur = cell(x_t, h_cur, mask=mask_t)
                output_steps.append(h_cur.unsqueeze(1))

            cur_layer_input = torch.cat(output_steps, dim=1)
            h_states[layer_idx] = h_cur
            if layer_idx < self.num_layers - 1:
                b_t = B * T
                dropped = self.dropout(cur_layer_input.view(b_t, self.hidden_dim, H, W))
                cur_layer_input = dropped.view(B, T, self.hidden_dim, H, W)

        return cur_layer_input, h_states

In [ ]:
# 2. Các module Chú ý & Cổ Giảm Kênh (SpatialReductionNeck & Dual Attention)
class SpatialReductionNeck(nn.Module):
    def __init__(self, in_channels: int = 448, out_channels: int = 64, dropout: float = 0.25):
        super().__init__()
        self.downsample_p3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.upsample_p5 = nn.Upsample(scale_factor=2, mode="nearest")
        self.project = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
            nn.Dropout2d(p=dropout) if dropout > 0 else nn.Identity()
        )

    def forward(self, p3: torch.Tensor, p4: torch.Tensor, p5: torch.Tensor) -> torch.Tensor:
        B, T = p3.shape[:2]
        bt = B * T

        p3_flat = p3.contiguous().view(bt, *p3.shape[2:])
        p4_flat = p4.contiguous().view(bt, *p4.shape[2:])
        p5_flat = p5.contiguous().view(bt, *p5.shape[2:])

        p3_scaled = self.downsample_p3(p3_flat)
        p4_scaled = p4_flat
        p5_scaled = self.upsample_p5(p5_flat)

        fused_flat = torch.cat([p3_scaled, p4_scaled, p5_scaled], dim=1)
        reduced_flat = self.project(fused_flat)
        return reduced_flat.view(B, T, *reduced_flat.shape[1:])


class SpatialAttentionPooling(nn.Module):
    def __init__(self, in_channels: int):
        super().__init__()
        self.attn_conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 2, kernel_size=1),
            nn.SiLU(inplace=True),
            nn.Conv2d(in_channels // 2, 1, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C, H, W = x.shape
        flat = x.view(B * T, C, H, W)
        scores = self.attn_conv(flat).view(B * T, 1, H * W)
        weights = F.softmax(scores, dim=-1)

        features_flat = flat.view(B * T, C, H * W)
        pooled = torch.bmm(features_flat, weights.transpose(1, 2)).squeeze(-1)
        return pooled.view(B, T, C)


class TemporalAttentionPooling(nn.Module):
    def __init__(self, feature_dim: int):
        super().__init__()
        self.score_net = nn.Sequential(
            nn.Linear(feature_dim, feature_dim // 2),
            nn.Tanh(),
            nn.Linear(feature_dim // 2, 1)
        )

    def forward(self, x: torch.Tensor, seq_lens: Optional[torch.Tensor] = None) -> torch.Tensor:
        scores = self.score_net(x).squeeze(-1)

        if seq_lens is not None:
            B, T = scores.shape
            t_idx = torch.arange(T, device=scores.device).unsqueeze(0)
            mask = t_idx >= seq_lens.unsqueeze(1)
            fill_value = -1e4 if scores.dtype == torch.float16 else -1e9
            scores = scores.masked_fill(mask, fill_value)

        weights = F.softmax(scores, dim=-1).unsqueeze(1)
        context = torch.bmm(weights, x).squeeze(1)
        return context


In [ ]:
# 3. Lớp Mô hình Hoàn chỉnh ConvGRUClassifier
class ConvGRUClassifier(nn.Module):
    def __init__(
        self,
        neck_channels: int = 64,
        hidden_dim: int = 192,
        num_layers: int = 2,
        num_classes: int = 2,
        dropout: float = 0.35,
        adapter_dropout: float = 0.25,
        supervision_mode: str = "attention_pooling"
    ):
        super().__init__()
        self.supervision_mode = supervision_mode
        self.num_classes = num_classes

        self.spatial_neck = SpatialReductionNeck(
            in_channels=448,
            out_channels=neck_channels,
            dropout=adapter_dropout
        )

        self.conv_gru = ConvGRU(
            input_dim=neck_channels,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            kernel_size=3,
            dropout=adapter_dropout
        )

        self.spatial_attn = SpatialAttentionPooling(in_channels=hidden_dim)
        self.temporal_attn = TemporalAttentionPooling(feature_dim=hidden_dim)

        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.SiLU(inplace=True),
            nn.Dropout(p=dropout / 2),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(
        self,
        features: Tuple[torch.Tensor, torch.Tensor, torch.Tensor],
        seq_lens: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        p3, p4, p5 = features
        spatial_feats = self.spatial_neck(p3, p4, p5)
        gru_out, _ = self.conv_gru(spatial_feats, seq_lens=seq_lens)
        spatial_pooled = self.spatial_attn(gru_out)
        temporal_pooled = self.temporal_attn(spatial_pooled, seq_lens=seq_lens)
        logits = self.classifier(temporal_pooled)
        return logits

    @classmethod
    def from_config(cls, cfg: KaggleTrainConfig) -> "ConvGRUClassifier":
        return cls(
            neck_channels=getattr(cfg, "spatial_neck_channels", 64),
            hidden_dim=cfg.hidden_dim,
            num_layers=cfg.num_layers,
            num_classes=cfg.num_classes,
            dropout=cfg.dropout,
            adapter_dropout=cfg.adapter_dropout,
            supervision_mode=cfg.supervision_mode
        )

## 📹 Section 5: Pipeline Nạp & Tiền xử lý Video Thô
- Tối ưu hóa bộ nhớ RAM với Tensor `torch.uint8` trên CPU.
- Giữ nguyên tỷ lệ khung hình video qua phép biến đổi `letterbox` ($640 \times 640$).
- Ghép batch động qua `collate_video_frames` với zero-padding theo $T_{max}$ của batch.

In [ ]:
# 1. Cài đặt các hàm và tiện ích tiền xử lý video
import subprocess
import shutil
import contextlib

@contextlib.contextmanager
def _suppress_c_stderr():
    """Tạm thời chuyển hướng file descriptor 2 (C/C++ stderr) sang os.devnull để chặn cảnh báo swscaler gây spam log."""
    try:
        devnull_fd = os.open(os.devnull, os.O_WRONLY)
        old_stderr_fd = os.dup(2)
        os.dup2(devnull_fd, 2)
        try:
            yield
        finally:
            os.dup2(old_stderr_fd, 2)
            os.close(old_stderr_fd)
            os.close(devnull_fd)
    except Exception:
        yield

# Hàm letterbox chuẩn hoá khung hình
def letterbox(image: np.ndarray, new_size: int = 640, color: Tuple[int, int, int] = (114, 114, 114)) -> np.ndarray:
    h, w = image.shape[:2]
    scale = min(new_size / h, new_size / w)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    top = (new_size - new_h) // 2
    bottom = new_size - new_h - top
    left = (new_size - new_w) // 2
    right = new_size - new_w - left

    padded = cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)
    return padded

# 2. Cấu trúc mẫu video và Dataset
@dataclass
class RawVideoSample:
    path: Path
    video_id: str
    split: str
    label: int
    label_name: str
    source_dataset: str = "custom"
    subject_id: Optional[str] = None

class RawVideoFramesDataset(Dataset):
    def __init__(
        self,
        dataset_dir: Union[str, Path],
        manifest_file: Optional[Union[str, Path]] = None,
        split: str = "train",
        sample_interval: float = 0.2,
        seq_len: Optional[int] = None,
        img_size: int = 640,
        min_frames: int = 10,
        use_augmentation: bool = True
    ):
        self.dataset_dir = self._resolve_dataset_dir(Path(dataset_dir))
        self.split = split.lower().strip()
        self.sample_interval = float(sample_interval)
        self.seq_len = seq_len
        self.img_size = img_size
        self.min_frames = min_frames
        self.use_augmentation = use_augmentation and (self.split == "train")

        self.samples = self._load_samples(manifest_file)

    def _resolve_dataset_dir(self, dataset_dir: Path) -> Path:
        """Tự động định vị thư mục dataset thực tế nếu đường dẫn cấu hình bị lệch trên Kaggle."""
        if dataset_dir.exists():
            return dataset_dir
        if Path("/kaggle/input").exists():
            for p in Path("/kaggle/input").rglob("dataset_merged_split.csv"):
                if p.is_file():
                    print(f"[*] Tự động tìm thấy dataset_dir từ manifest: {p.parent}")
                    return p.parent
            for p in Path("/kaggle/input").rglob("train"):
                if p.is_dir():
                    print(f"[*] Tự động tìm thấy dataset_dir từ thư mục train/: {p.parent}")
                    return p.parent
        return dataset_dir

    def _resolve_manifest_path(self, manifest_file: Optional[Union[str, Path]]) -> Optional[Path]:
        """Tìm kiếm file manifest CSV linh hoạt từ nhiều vị trí ứng viên."""
        candidates = []
        if manifest_file:
            p = Path(manifest_file)
            if p.is_file():
                return p
            candidates.extend([
                self.dataset_dir / p,
                self.dataset_dir.parent / p,
                self.dataset_dir / "data_processed" / p
            ])
        candidates.extend([
            self.dataset_dir / "dataset_merged_split.csv",
            self.dataset_dir.parent / "dataset_merged_split.csv",
            self.dataset_dir / "data_processed" / "dataset_merged_split.csv",
            self.dataset_dir / "dataset_manifest.csv"
        ])
        for c in candidates:
            if c.is_file():
                return c
        if Path("/kaggle/input").exists():
            for p in Path("/kaggle/input").rglob("dataset_merged_split.csv"):
                if p.is_file():
                    return p
            for p in Path("/kaggle/input").rglob("*.csv"):
                if p.is_file() and ("manifest" in p.name.lower() or "dataset" in p.name.lower()):
                    return p
        return None

    def _load_samples(self, manifest_file: Optional[Union[str, Path]]) -> List[RawVideoSample]:
        samples: List[RawVideoSample] = []
        manifest_path = self._resolve_manifest_path(manifest_file)

        if manifest_path and manifest_path.is_file():
            with open(manifest_path, mode="r", encoding="utf-8-sig") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    row_split = (row.get("split") or "train").strip().lower()
                    if row_split != self.split:
                        continue
                    
                    # Hỗ trợ toàn diện các tên cột đường dẫn phổ biến
                    rel_p = (
                        row.get("processed_path") or
                        row.get("rel_path") or
                        row.get("relative_path") or
                        row.get("video_path") or
                        row.get("path") or
                        row.get("file_path") or
                        row.get("orig_file") or
                        row.get("filename")
                    )
                    if not rel_p:
                        continue

                    # Tìm kiếm vị trí thực tế của tệp video
                    cand_paths = [
                        self.dataset_dir / rel_p,
                        self.dataset_dir / "data_processed" / rel_p,
                        self.dataset_dir.parent / rel_p,
                        Path(rel_p),
                        self.dataset_dir / Path(rel_p).name,
                        self.dataset_dir / self.split / Path(rel_p).name
                    ]
                    v_path = None
                    for cp in cand_paths:
                        if cp.is_file():
                            v_path = cp
                            break
                    if v_path is None:
                        continue

                    # Lấy nhãn: ưu tiên binary_label (0 hoặc 1), sau đó đến label
                    lbl_str = row.get("binary_label") if row.get("binary_label") is not None else row.get("label", row.get("class", 0))
                    try:
                        lbl_val = int(float(lbl_str))
                    except (ValueError, TypeError):
                        lbl_val = 0

                    v_id = row.get("clip_id") or row.get("video_id") or row.get("source_video_id") or v_path.stem
                    lbl_name = row.get("label_name") or ("0_alert" if lbl_val == 0 else "1_drowsy")
                    samples.append(RawVideoSample(
                        path=v_path, video_id=v_id, split=row_split,
                        label=lbl_val, label_name=lbl_name,
                        source_dataset=row.get("source_dataset", "custom"),
                        subject_id=row.get("subject_id")
                    ))
            print(f"[✓] [{self.split.upper()}] Đã nạp thành công {len(samples)} mẫu từ manifest: {manifest_path}")

        # Quét trực tiếp thư mục nếu không có manifest hoặc manifest không khớp video nào
        if len(samples) == 0:
            target_dir = self.dataset_dir / self.split if (self.dataset_dir / self.split).is_dir() else self.dataset_dir
            if target_dir.exists():
                for v_path in target_dir.rglob("*"):
                    if v_path.suffix.lower() in (".mp4", ".avi", ".mkv", ".mov"):
                        parent_name = v_path.parent.name.lower()
                        lbl = 1 if ("drowsy" in parent_name or "sleep" in parent_name or "1" in parent_name) else 0
                        samples.append(RawVideoSample(
                            path=v_path, video_id=v_path.stem, split=self.split,
                            label=lbl, label_name=f"{lbl}_{'drowsy' if lbl == 1 else 'alert'}"
                        ))
                if len(samples) > 0:
                    print(f"[*] [{self.split.upper()}] Quét trực tiếp thư mục {target_dir}, tìm thấy {len(samples)} video.")

        return samples

    def _sample_video_frames_ffmpeg(self, video_path: Path) -> List[np.ndarray]:
        """
        Dùng FFmpeg CLI (có sẵn trên Kaggle Linux) để giải mã và deinterlace (khử xen kẽ)
        các video có định dạng yuv411p / DV / interlaced mà OpenCV swscaler báo lỗi:
        'Cannot convert interlaced to progressive frames or vice versa'.
        """
        ffmpeg_bin = shutil.which("ffmpeg") or ("/usr/bin/ffmpeg" if os.path.exists("/usr/bin/ffmpeg") else None)
        if not ffmpeg_bin:
            return []

        fps_sample = max(0.1, 1.0 / self.sample_interval)
        # 1. yadif=0:-1:0: khử xen kẽ (deinterlace) các trường thành progressive frames hoàn chỉnh
        # 2. fps: lấy mẫu đúng chu kỳ (ví dụ: sample_interval=0.2s -> fps=5)
        # 3. scale: resize giữ tỷ lệ gốc, không vượt quá self.img_size
        # 4. format=rgb24: chuyển không gian màu sang RGB24 trước khi pad viền
        # 5. pad: đệm viền xám (114, 114, 114) thành đúng kích thước vuông self.img_size x self.img_size (chuẩn letterbox)
        vf = (
            f"yadif=0:-1:0,"
            f"fps={fps_sample},"
            f"scale={self.img_size}:{self.img_size}:force_original_aspect_ratio=decrease,"
            f"format=rgb24,"
            f"pad={self.img_size}:{self.img_size}:(ow-iw)/2:(oh-ih)/2:color=0x727272"
        )
        cmd = [
            str(ffmpeg_bin),
            "-nostdin",
            "-hide_banner",
            "-loglevel", "error",
            "-i", str(video_path),
            "-vf", vf,
            "-f", "rawvideo",
            "-pix_fmt", "rgb24",
            "pipe:1"
        ]
        try:
            proc = subprocess.run(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                timeout=60,
                check=False
            )
            if proc.returncode != 0:
                return []
            raw_bytes = proc.stdout
            frame_len = self.img_size * self.img_size * 3
            n_frames = len(raw_bytes) // frame_len
            if n_frames == 0:
                return []
            arr = np.frombuffer(raw_bytes[: n_frames * frame_len], dtype=np.uint8)
            frames = arr.reshape(n_frames, self.img_size, self.img_size, 3)
            return [frames[i] for i in range(n_frames)]
        except Exception:
            return []

    def _sample_video_frames(self, video_path: Path) -> List[np.ndarray]:
        frames_rgb: List[np.ndarray] = []
        cap = cv2.VideoCapture(str(video_path))
        if cap.isOpened():
            try:
                fps = cap.get(cv2.CAP_PROP_FPS)
                if not fps or fps <= 0 or math.isnan(fps):
                    fps = 30.0
                step = max(1, int(round(fps * self.sample_interval)))

                frame_idx = 0
                with _suppress_c_stderr():
                    while True:
                        ret, frame_bgr = cap.read()
                        if not ret or frame_bgr is None:
                            break
                        if frame_idx % step == 0:
                            frame_lb = letterbox(frame_bgr, new_size=self.img_size)
                            frame_rgb = cv2.cvtColor(frame_lb, cv2.COLOR_BGR2RGB)
                            frames_rgb.append(frame_rgb)
                        frame_idx += 1
            except Exception:
                frames_rgb = []
            finally:
                cap.release()

        # Fallback sang FFmpeg nếu OpenCV không đọc được hoặc không đủ số lượng frame tối thiểu
        # (đặc biệt đối với video interlaced yuv411p mà OpenCV swscaler không chuyển đổi được)
        if len(frames_rgb) < self.min_frames:
            ffmpeg_frames = self._sample_video_frames_ffmpeg(video_path)
            if len(ffmpeg_frames) >= self.min_frames:
                return ffmpeg_frames

        return frames_rgb

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[Optional[torch.Tensor], torch.Tensor, torch.Tensor, Dict[str, Any]]:
        sample = self.samples[idx]
        frames_rgb = self._sample_video_frames(sample.path)

        if len(frames_rgb) < self.min_frames:
            meta = {"video_id": sample.video_id, "status": "corrupted"}
            return None, torch.tensor(sample.label), torch.tensor(0), meta

        if self.seq_len is not None and len(frames_rgb) > self.seq_len:
            start_idx = random.randint(0, len(frames_rgb) - self.seq_len) if self.split == "train" else (len(frames_rgb) - self.seq_len) // 2
            frames_rgb = frames_rgb[start_idx : start_idx + self.seq_len]

        if self.use_augmentation and random.random() < 0.5:
            frames_rgb = [cv2.flip(f, 1) for f in frames_rgb]

        batch_np = np.stack(frames_rgb, axis=0).transpose(0, 3, 1, 2)
        frames_t = torch.from_numpy(batch_np).to(dtype=torch.uint8)
        label_t = torch.tensor(sample.label, dtype=torch.long)
        seq_len_t = torch.tensor(len(frames_rgb), dtype=torch.long)
        meta = {"video_id": sample.video_id, "status": "ok"}
        return frames_t, label_t, seq_len_t, meta

# 3. Hàm gom batch và Factory DataLoader
def collate_video_frames(
    batch: List[Tuple[Optional[torch.Tensor], torch.Tensor, torch.Tensor, Dict[str, Any]]]
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, List[Dict[str, Any]]]:
    """
    Gom cụm batch các video có chiều dài thời gian biến thiên với dynamic zero-padding.
    Trả về:
      - b_frames: [B, T_max, 3, H, W] uint8 tensor
      - b_labels: [B] long tensor
      - b_seq_lens: [B] long tensor (số khung hình hợp lệ của từng video)
      - b_metas: List danh sách metadata
    """
    valid_batch = [item for item in batch if item[0] is not None]
    if len(valid_batch) == 0:
        return torch.empty(0), torch.empty(0), torch.empty(0), []

    seq_lens = [item[2].item() for item in valid_batch]
    t_max = max(seq_lens)
    _, _, H, W = valid_batch[0][0].shape
    batch_size = len(valid_batch)

    b_frames = torch.zeros(batch_size, t_max, 3, H, W, dtype=torch.uint8)
    for i, item in enumerate(valid_batch):
        f_t, _, s_len, _ = item
        t_i = s_len.item()
        b_frames[i, :t_i] = f_t[:t_i]

    b_labels = torch.stack([item[1] for item in valid_batch])
    b_seq_lens = torch.tensor(seq_lens, dtype=torch.long)
    b_metas = [item[3] for item in valid_batch]
    return b_frames, b_labels, b_seq_lens, b_metas

def dataloader_worker_init_fn(worker_id: int) -> None:
    """Tắt đa luồng cục bộ của OpenCV trong DataLoader worker để tránh deadlock và tràn /dev/shm trên Kaggle Linux."""
    import cv2
    cv2.setNumThreads(0)

def build_dataloaders(cfg: KaggleTrainConfig) -> Tuple[DataLoader, DataLoader]:
    train_ds = RawVideoFramesDataset(
        dataset_dir=cfg.dataset_dir,
        manifest_file=cfg.manifest_file,
        split="train",
        sample_interval=cfg.sample_interval,
        seq_len=cfg.seq_len,
        img_size=cfg.image_size[0],
        min_frames=cfg.min_frames,
        use_augmentation=cfg.use_augmentation
    )
    val_ds = RawVideoFramesDataset(
        dataset_dir=cfg.dataset_dir,
        manifest_file=cfg.manifest_file,
        split="val",
        sample_interval=cfg.sample_interval,
        seq_len=cfg.seq_len,
        img_size=cfg.image_size[0],
        min_frames=cfg.min_frames,
        use_augmentation=False
    )

    if len(train_ds) == 0:
        available_inputs = []
        if Path("/kaggle/input").exists():
            for p in Path("/kaggle/input").rglob("*"):
                if p.is_dir() or p.suffix in (".csv", ".mp4", ".avi", ".pt"):
                    available_inputs.append(str(p))
                if len(available_inputs) >= 20:
                    break
        err_msg = (
            f"\n" + "=" * 80 + "\n"
            f"[!] LỖI KHỞI TẠO DATALOADER: Không tìm thấy bất kỳ video nào cho tập Train (0 mẫu)!\n"
            f"    - Thư mục cfg.dataset_dir: '{cfg.dataset_dir}' (Tồn tại: {Path(cfg.dataset_dir).exists()})\n"
            f"    - Tệp cfg.manifest_file: '{cfg.manifest_file}'\n"
            f"    - Các mục tìm thấy trong /kaggle/input: {available_inputs}\n"
            f"    -> HƯỚNG DẪN KHẮC PHỤC:\n"
            f"       1. Kiểm tra tên Dataset bạn đã đính kèm trên Kaggle Input.\n"
            f"       2. Chỉnh lại 'dataset_dir' trong Cell 4 cho trỏ đúng vào thư mục đó.\n"
            f"=" * 80
        )
        raise RuntimeError(err_msg)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=cfg.shuffle,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        collate_fn=collate_video_frames,
        worker_init_fn=dataloader_worker_init_fn if cfg.num_workers > 0 else None
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.val_batch_size,
        shuffle=False,
        num_workers=cfg.val_num_workers,
        pin_memory=cfg.pin_memory,
        collate_fn=collate_video_frames,
        worker_init_fn=dataloader_worker_init_fn if cfg.val_num_workers > 0 else None
    )
    return train_loader, val_loader


## 🛠️ Section 6: Hạ tầng Huấn luyện & Cơ chế Tiếp tục Huấn luyện (Resume Training)
- **EarlyStopping**: Dừng sớm khi `val_f1` bão hòa qua $patience=10$ epochs.
- **Compute Loss & Preds**: Chuẩn hóa gradient giữa `CrossEntropyLoss` (2-class) và `BCEWithLogitsLoss` (1-class).
- **Cơ chế Khôi phục Checkpoint Toàn diện**:
  - `load_checkpoint`: Phân giải thông minh số nguyên epoch (`5`), bí danh (`'last'`, `'best'`), hoặc đường dẫn tệp trực tiếp.
  - Bảo toàn trọn vẹn: `model_state_dict`, `optimizer_state_dict`, `scheduler_state_dict`, `scaler_state_dict` (AMP FP16), và `best_val_f1`.
  - Tự động dọn dẹp xoay vòng (`max_keep_ckpts = 5`) tránh tràn đĩa `/kaggle/working` 20GB.
  - Bảo toàn lịch sử `training_history.csv` khi resume.

In [ ]:
# 1. Hàm tính toán chỉ số đánh giá & Lớp EarlyStopping
def calculate_metrics(preds: np.ndarray, targets: np.ndarray) -> Dict[str, float]:
    if len(preds) == 0 or len(targets) == 0:
        return {"acc": 0.0, "recall": 0.0, "f1": 0.0}
    acc = accuracy_score(targets, preds)
    recall = recall_score(targets, preds, average='macro', zero_division=0)
    f1 = f1_score(targets, preds, average='macro', zero_division=0)
    return {"acc": float(acc), "recall": float(recall), "f1": float(f1)}

class EarlyStopping:
    def __init__(self, patience: int = 10, min_delta: float = 1e-4, mode: str = "max", monitor: str = "val_f1"):
        self.patience = max(1, patience)
        self.min_delta = float(min_delta)
        self.mode = mode.lower().strip()
        self.monitor = monitor
        self.best_score = -float("inf") if self.mode == "max" else float("inf")
        self.counter = 0
        self.best_epoch = 0
        self.early_stop = False

    def step(self, current_val: float, epoch: int) -> bool:
        improved = (current_val > self.best_score + self.min_delta) if self.mode == "max" else (current_val < self.best_score - self.min_delta)
        if improved:
            self.best_score = current_val
            self.best_epoch = epoch
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop

# 2. Pipeline Huấn luyện Chuyên biệt KaggleTrainer (Hỗ trợ Resume toàn diện & An toàn tuyệt đối)
class KaggleTrainer:
    def __init__(self, config: KaggleTrainConfig):
        self.cfg = config
        self.device = torch.device(self.cfg.device if torch.cuda.is_available() else "cpu")
        print(f"[*] Thiết bị huấn luyện : {self.device}")

        # Khởi tạo Dataloader
        print("[*] Đang khởi tạo DataLoaders...")
        self.train_loader, self.val_loader = build_dataloaders(self.cfg)
        print(f"[*] Tập Train: {len(self.train_loader.dataset)} videos | Tập Val: {len(self.val_loader.dataset)} videos")

        # Khởi tạo Extractor GPU Mini-Chunk
        print("[*] Đang nạp BackboneNeck PAFPN Extractor...")
        self.extractor = ChunkedBackboneNeckExtractor(
            checkpoint_path=self.cfg.backbone_neck_checkpoint,
            device=self.cfg.device,
            chunk_size=self.cfg.chunk_size,
            use_fp16=self.cfg.amp,
            img_size=self.cfg.image_size[0]
        ).to(self.device)

        # Khởi tạo Model chính
        print("[*] Đang nạp ConvGRUClassifier...")
        self.model = ConvGRUClassifier.from_config(self.cfg).to(self.device)

        # Cấu hình Loss
        self.is_binary_bce = (self.cfg.loss_type.lower() == "bce" and self.cfg.num_classes == 1)
        if self.is_binary_bce:
            pos_w = torch.tensor([self.cfg.pos_weight], device=self.device) if self.cfg.pos_weight else None
            self.criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w, reduction=self.cfg.bce_reduction)
        else:
            self.criterion = nn.CrossEntropyLoss()

        # Optimizer & Scheduler
        self.optimizer = optim.AdamW(
            self.model.parameters(),
            lr=self.cfg.lr0,
            betas=self.cfg.betas,
            weight_decay=self.cfg.weight_decay
        )
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer,
            T_max=self.cfg.epochs,
            eta_min=self.cfg.lr0 * self.cfg.lr_min_factor
        ) if self.cfg.use_scheduler else None

        # AMP Scaler
        self.scaler = torch.amp.GradScaler("cuda") if (self.cfg.amp and self.device.type == "cuda") else None

        # Checkpoints & Logs
        self.ckpt_dir = Path(self.cfg.checkpoint_dir)
        self.ckpt_dir.mkdir(parents=True, exist_ok=True)
        self.history_csv_path = Path(self.cfg.history_csv_path)
        self._init_history_csv()

        # TensorBoard SummaryWriter & Bộ đếm bước toàn cục
        self.tb_dir = Path(self.cfg.tb_log_dir) / self.cfg.experiment_name
        self.tb_dir.mkdir(parents=True, exist_ok=True)
        self.writer = SummaryWriter(log_dir=str(self.tb_dir))
        self.global_step = 0
        print(f"[*] TensorBoard SummaryWriter khởi tạo tại: {self.tb_dir}")

        # Early Stopping
        self.early_stopper = EarlyStopping(
            patience=self.cfg.patience,
            min_delta=self.cfg.min_delta,
            mode=self.cfg.monitor_mode,
            monitor=self.cfg.monitor_metric
        ) if self.cfg.early_stopping else None

        self.best_val_f1 = 0.0
        self.start_epoch = 1

        # Tự động Resume nếu được kích hoạt trong config
        if getattr(self.cfg, "enable_resume", False) and getattr(self.cfg, "resume", None):
            self.load_checkpoint(self.cfg.resume)

    def _init_history_csv(self) -> None:
        self.history_csv_path.parent.mkdir(parents=True, exist_ok=True)
        if not self.history_csv_path.exists():
            with open(self.history_csv_path, mode="w", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow(["epoch", "train_loss", "train_acc", "train_recall", "train_f1",
                                 "val_loss", "val_acc", "val_recall", "val_f1", "lr", "time_sec"])

    def _atomic_save(self, state: Dict[str, Any], target_path: Path) -> None:
        """Lưu file checkpoint theo cơ chế Atomic Rename để chống hỏng file khi bị ngắt đột ngột."""
        target_path.parent.mkdir(parents=True, exist_ok=True)
        tmp_path = target_path.with_suffix(f"{target_path.suffix}.tmp")
        torch.save(state, tmp_path)
        os.replace(tmp_path, target_path)

    def _resolve_checkpoint_path(self, checkpoint_path: Union[str, Path, int]) -> Path:
        """
        Phân giải đường dẫn checkpoint linh hoạt:
          - Số nguyên hoặc chuỗi số (VD: 5 hoặc '5') -> tìm 'epoch_5.pt'
          - Bí danh 'last' -> 'last.pt'
          - Bí danh 'best' -> 'best.pt'
          - Đường dẫn file cụ thể
        """
        # 1. Trường hợp số nguyên hoặc chuỗi số đại diện cho epoch
        if isinstance(checkpoint_path, int) or (isinstance(checkpoint_path, str) and checkpoint_path.strip().isdigit()):
            ep_num = int(checkpoint_path)
            candidate = self.ckpt_dir / f"epoch_{ep_num}.pt"
            if candidate.is_file():
                return candidate
            for search_root in [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]:
                if search_root.exists():
                    found = list(search_root.rglob(f"epoch_{ep_num}.pt"))
                    if found:
                        return found[0]
            raise FileNotFoundError(f"Không tìm thấy checkpoint cho epoch {ep_num} (đã thử tìm 'epoch_{ep_num}.pt')")

        cp_str = str(checkpoint_path).strip().lower()
        # 2. Trường hợp bí danh 'last'
        if cp_str in ("last", "last.pt"):
            target = self.ckpt_dir / "last.pt"
            if not target.is_file() and Path("/kaggle/input").exists():
                found = list(Path("/kaggle/input").rglob("last.pt"))
                if found:
                    return found[0]
            return target

        # 3. Trường hợp bí danh 'best'
        if cp_str in ("best", "best.pt"):
            target = self.ckpt_dir / "best.pt"
            if not target.is_file() and Path("/kaggle/input").exists():
                found = list(Path("/kaggle/input").rglob("best.pt"))
                if found:
                    return found[0]
            return target

        # 4. Trường hợp đường dẫn tệp cụ thể
        p = Path(checkpoint_path)
        if p.is_file():
            return p
        cand = self.ckpt_dir / p.name
        if cand.is_file():
            return cand

        raise FileNotFoundError(f"Không tìm thấy tệp checkpoint tại: {checkpoint_path}")

    def load_checkpoint(self, checkpoint_path: Union[str, Path, int]) -> None:
        """Khôi phục trọn vẹn trạng thái huấn luyện từ checkpoint."""
        target_path = self._resolve_checkpoint_path(checkpoint_path)
        print(f"[*] Đang nạp checkpoint khôi phục: {target_path}")
        if not target_path.exists():
            raise FileNotFoundError(f"Không tìm thấy checkpoint tại: {target_path}")

        ckpt = torch.load(target_path, map_location=self.device, weights_only=False)

        # 1. Khôi phục trọng số mô hình và Optimizer
        self.model.load_state_dict(ckpt["model_state_dict"])
        self.optimizer.load_state_dict(ckpt["optimizer_state_dict"])

        # 2. Khôi phục Lịch trình học Cosine Scheduler
        if self.scheduler is not None and "scheduler_state_dict" in ckpt and ckpt["scheduler_state_dict"] is not None:
            self.scheduler.load_state_dict(ckpt["scheduler_state_dict"])

        # 3. Khôi phục trạng thái GradScaler (AMP FP16)
        if self.scaler is not None and "scaler_state_dict" in ckpt and ckpt["scaler_state_dict"] is not None:
            self.scaler.load_state_dict(ckpt["scaler_state_dict"])
            print("[*] Đã khôi phục thành công trạng thái GradScaler (AMP).")

        # 4. Khôi phục best F1 và epoch bắt đầu
        self.best_val_f1 = ckpt.get("best_val_f1", 0.0)
        saved_epoch = ckpt.get("epoch", 0)

        # 5. Khôi phục an toàn global_step với cơ chế Fallback mặc định
        if "global_step" in ckpt and ckpt["global_step"] is not None:
            self.global_step = int(ckpt["global_step"])
            print(f"    - Global Step khôi phục   : {self.global_step}")
        else:
            # Cơ chế Fallback an toàn: Nếu checkpoint cũ không có 'global_step',
            # tự động ước lượng giá trị mặc định từ (saved_epoch * num_batches) hoặc 0
            num_batches = len(self.train_loader) if (hasattr(self, "train_loader") and self.train_loader is not None) else 0
            default_step = max(0, saved_epoch * num_batches)
            self.global_step = default_step
            print(f"    - [LƯU Ý] Checkpoint không chứa 'global_step'. Tự động đặt về giá trị mặc định: {self.global_step}")

        # 6. Đồng bộ trạng thái EarlyStopping
        if self.early_stopper is not None:
            self.early_stopper.best_score = self.best_val_f1
            self.early_stopper.best_epoch = saved_epoch
            self.early_stopper.counter = 0

        # 7. Đồng bộ lịch sử training_history.csv nếu tồn tại ở thư mục checkpoint nguồn
        src_history = target_path.parent / "training_history.csv"
        if src_history.exists() and src_history.resolve() != self.history_csv_path.resolve():
            try:
                import pandas as pd
                df_src = pd.read_csv(src_history)
                df_src = df_src[df_src["epoch"] <= saved_epoch]
                df_src.to_csv(self.history_csv_path, index=False)
                print(f"[✓] Đã phục hồi lịch sử huấn luyện từ {src_history} (đến epoch {saved_epoch}).")
            except Exception as e:
                print(f"[!] Không thể đồng bộ lịch sử CSV: {e}")

        if self.cfg.resume_epoch is not None:
            self.start_epoch = int(self.cfg.resume_epoch)
        else:
            self.start_epoch = saved_epoch + 1

        cur_lr = self.optimizer.param_groups[0]['lr']
        print(f"[✓] KHÔI PHỤC THÀNH CÔNG! Sẽ tiếp tục huấn luyện từ Epoch {self.start_epoch}")
        print(f"    - Epoch đã lưu trong ckpt: {saved_epoch}")
        print(f"    - Best Val F1 trước đó   : {self.best_val_f1:.4f}")
        print(f"    - Learning Rate hiện tại : {cur_lr:.2e}")

    def _prune_old_checkpoints(self) -> None:
        """Dọn dẹp các checkpoint epoch cũ, chỉ giữ lại max_keep_ckpts file gần nhất để tránh đầy đĩa Kaggle."""
        if self.cfg.max_keep_ckpts <= 0:
            return
        epoch_files = []
        for p in self.ckpt_dir.glob("epoch_*.pt"):
            try:
                ep_num = int(p.stem.replace("epoch_", ""))
                epoch_files.append((ep_num, p))
            except ValueError:
                continue
        epoch_files.sort(key=lambda x: x[0])
        if len(epoch_files) > self.cfg.max_keep_ckpts:
            to_remove = epoch_files[:-self.cfg.max_keep_ckpts]
            for ep_num, p in to_remove:
                try:
                    p.unlink()
                except Exception:
                    pass

    def save_checkpoint(self, epoch: int, is_best: bool, val_f1: Optional[float] = None) -> None:
        state = {
            "epoch": epoch,
            "global_step": self.global_step,
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "scheduler_state_dict": self.scheduler.state_dict() if self.scheduler is not None else None,
            "scaler_state_dict": self.scaler.state_dict() if self.scaler is not None else None,
            "best_val_f1": self.best_val_f1,
            "val_f1": val_f1
        }
        # 1. Luôn lưu last.pt nguyên tử
        self._atomic_save(state, self.ckpt_dir / "last.pt")

        # 2. Lưu best.pt khi đạt kỷ lục F1 nguyên tử
        if is_best:
            self._atomic_save(state, self.ckpt_dir / "best.pt")
            print(f" [★ BEST F1: {val_f1:.4f}] Đã lưu checkpoint best.pt")

        # 3. Lưu epoch_{epoch}.pt theo định kỳ nếu cho phép
        if not self.cfg.save_best_only:
            interval = max(1, self.cfg.save_epoch_interval)
            if epoch % interval == 0:
                epoch_path = self.ckpt_dir / f"epoch_{epoch}.pt"
                self._atomic_save(state, epoch_path)
                self._prune_old_checkpoints()

    def compute_loss_and_preds(self, logits: torch.Tensor, labels: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        if self.is_binary_bce:
            logits_1d = logits.view(-1)
            loss = self.criterion(logits_1d, labels.float().view(-1))
            preds = (torch.sigmoid(logits_1d) >= 0.5).long()
        else:
            loss = self.criterion(logits, labels.long())
            preds = torch.argmax(logits, dim=1)
        return loss, preds

    def train_epoch(self, epoch: int) -> Dict[str, float]:
        self.model.train()
        total_loss = 0.0
        all_preds, all_targets = [], []
        valid_batches_count = 0
        accum_steps = max(1, self.cfg.gradient_accumulation_steps)
        accum_count = 0
        self.optimizer.zero_grad()

        num_batches = len(self.train_loader)
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch:02d}/{self.cfg.epochs:02d} [Train]", leave=False)
        for batch_idx, batch in enumerate(pbar):
            frames, labels, seq_lens, _ = batch
            if frames.numel() == 0:
                # Nếu batch rỗng rơi vào batch cuối cùng của epoch nhưng vẫn còn gradient tích lũy dở
                if (batch_idx + 1) == num_batches and accum_count > 0:
                    if self.scaler is not None:
                        if self.cfg.grad_clip_norm > 0:
                            self.scaler.unscale_(self.optimizer)
                            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                        self.scaler.step(self.optimizer)
                        self.scaler.update()
                    else:
                        if self.cfg.grad_clip_norm > 0:
                            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                        self.optimizer.step()
                    self.optimizer.zero_grad()
                    accum_count = 0
                continue

            labels = labels.to(self.device, non_blocking=True)
            seq_lens = seq_lens.to(self.device, non_blocking=True)

            try:
                p3, p4, p5 = self.extractor(frames)

                if self.scaler is not None:
                    with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                        logits = self.model((p3, p4, p5), seq_lens=seq_lens)
                        raw_loss, preds = self.compute_loss_and_preds(logits, labels)
                        loss = raw_loss / accum_steps
                    self.scaler.scale(loss).backward()
                else:
                    logits = self.model((p3, p4, p5), seq_lens=seq_lens)
                    raw_loss, preds = self.compute_loss_and_preds(logits, labels)
                    loss = raw_loss / accum_steps
                    loss.backward()

                accum_count += 1
                total_loss += raw_loss.item()
                valid_batches_count += 1
                preds_np = preds.detach().cpu().numpy()
                targets_np = labels.detach().cpu().numpy()
                all_preds.extend(preds_np)
                all_targets.extend(targets_np)

                self.global_step += 1

                # Ghi log TensorBoard cấp độ từng Step (Per-Step Metrics)
                if self.cfg.enable_step_logging and self.global_step % self.cfg.log_step_interval == 0:
                    step_metrics = calculate_metrics(preds_np, targets_np)
                    self.writer.add_scalar("Train_Step/Loss", raw_loss.item(), self.global_step)
                    self.writer.add_scalar("Train_Step/Accuracy", step_metrics["acc"], self.global_step)
                    self.writer.add_scalar("Train_Step/Recall", step_metrics["recall"], self.global_step)
                    self.writer.add_scalar("Train_Step/F1", step_metrics["f1"], self.global_step)

                if self.global_step % getattr(self.cfg, "flush_step_interval", 50) == 0:
                    self.writer.flush()

                is_step_boundary = (accum_count == accum_steps) or ((batch_idx + 1) == num_batches and accum_count > 0)
                if is_step_boundary:
                    if self.scaler is not None:
                        if self.cfg.grad_clip_norm > 0:
                            self.scaler.unscale_(self.optimizer)
                            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                        self.scaler.step(self.optimizer)
                        self.scaler.update()
                    else:
                        if self.cfg.grad_clip_norm > 0:
                            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                        self.optimizer.step()
                    self.optimizer.zero_grad()
                    accum_count = 0

                pbar.set_postfix({"loss": f"{raw_loss.item():.4f}"})

            except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
                if "out of memory" in str(e).lower():
                    print(f"\n[!] Bắt ngoại lệ CUDA OOM tại Batch {batch_idx}. Giải phóng bộ nhớ...")
                    self.optimizer.zero_grad()
                    accum_count = 0
                    try:
                        del frames, labels, seq_lens, p3, p4, p5, logits, raw_loss, loss
                    except Exception:
                        pass
                    del e
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    continue
                else:
                    raise e

        metrics = calculate_metrics(np.array(all_preds), np.array(all_targets))
        metrics["loss"] = total_loss / max(1, valid_batches_count)
        return metrics

    def validate_epoch(self, epoch: int) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
        self.model.eval()
        total_loss = 0.0
        all_preds, all_targets = [], []
        valid_batches_count = 0

        with torch.no_grad():
            pbar = tqdm(self.val_loader, desc=f"Epoch {epoch:02d}/{self.cfg.epochs:02d} [Val]  ", leave=False)
            for batch_idx, batch in enumerate(pbar):
                frames, labels, seq_lens, _ = batch
                if frames.numel() == 0:
                    continue
                labels = labels.to(self.device, non_blocking=True)
                seq_lens = seq_lens.to(self.device, non_blocking=True)

                try:
                    p3, p4, p5 = self.extractor(frames)
                    if self.scaler is not None:
                        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                            logits = self.model((p3, p4, p5), seq_lens=seq_lens)
                            loss, preds = self.compute_loss_and_preds(logits, labels)
                    else:
                        logits = self.model((p3, p4, p5), seq_lens=seq_lens)
                        loss, preds = self.compute_loss_and_preds(logits, labels)

                    total_loss += loss.item()
                    valid_batches_count += 1
                    all_preds.extend(preds.cpu().numpy())
                    all_targets.extend(labels.cpu().numpy())
                except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
                    if "out of memory" in str(e).lower():
                        print(f"\n[!] Bắt ngoại lệ CUDA OOM tại Validation Batch {batch_idx}. Giải phóng bộ nhớ...")
                        try:
                            del frames, labels, seq_lens, p3, p4, p5, logits, loss, preds
                        except Exception:
                            pass
                        del e
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
                        continue
                    else:
                        raise e

        metrics = calculate_metrics(np.array(all_preds), np.array(all_targets))
        metrics["loss"] = total_loss / max(1, valid_batches_count)
        return metrics, np.array(all_preds), np.array(all_targets)

    def train(self) -> None:
        if self.start_epoch > self.cfg.epochs:
            print(f"[*] Mô hình đã huấn luyện hoàn tất đến epoch {self.start_epoch - 1} (Cấu hình tổng epochs={self.cfg.epochs}). Không cần huấn luyện thêm!")
            return

        print()
        print("=" * 75)
        print(f"  BẮT ĐẦU HUẤN LUYỆN SPATIOTEMPORAL CONVGRU (TỪ EPOCH {self.start_epoch:02d} ĐẾN {self.cfg.epochs:02d})")
        print("=" * 75)

        try:
            for epoch in range(self.start_epoch, self.cfg.epochs + 1):
                t_start = time.time()
                train_m = self.train_epoch(epoch)
                lr = self.optimizer.param_groups[0]['lr']

                # Ghi log Train Epoch vào TensorBoard
                self.writer.add_scalar("Train_Epoch/Loss", train_m["loss"], epoch)
                self.writer.add_scalar("Train_Epoch/Accuracy", train_m["acc"], epoch)
                self.writer.add_scalar("Train_Epoch/Recall", train_m["recall"], epoch)
                self.writer.add_scalar("Train_Epoch/F1", train_m["f1"], epoch)
                self.writer.add_scalar("Train_Epoch/LearningRate", lr, epoch)

                # Đánh giá tập Val
                val_m, preds, targets = None, None, None
                if len(self.val_loader.dataset) > 0 and (epoch % self.cfg.val_interval_epochs == 0 or epoch == self.cfg.epochs):
                    val_m, preds, targets = self.validate_epoch(epoch)
                    self.writer.add_scalar("Val_Epoch/Loss", val_m["loss"], epoch)
                    self.writer.add_scalar("Val_Epoch/Accuracy", val_m["acc"], epoch)
                    self.writer.add_scalar("Val_Epoch/Recall", val_m["recall"], epoch)
                    self.writer.add_scalar("Val_Epoch/F1", val_m["f1"], epoch)

                self.writer.flush()
                elapsed = time.time() - t_start

                is_best = False
                if val_m is not None:
                    if val_m["f1"] > self.best_val_f1:
                        self.best_val_f1 = val_m["f1"]
                        is_best = True

                    print(f"Epoch {epoch:02d}/{self.cfg.epochs:02d} | Train Loss: {train_m['loss']:.4f} - F1: {train_m['f1']:.4f} | "
                          f"Val Loss: {val_m['loss']:.4f} - F1: {val_m['f1']:.4f} - Acc: {val_m['acc']:.4f} | "
                          f"LR: {lr:.2e} | {elapsed:.1f}s", end="")
                else:
                    print(f"Epoch {epoch:02d}/{self.cfg.epochs:02d} | Train Loss: {train_m['loss']:.4f} - F1: {train_m['f1']:.4f} | "
                          f"(Val bỏ qua) | {elapsed:.1f}s", end="")

                self.save_checkpoint(epoch, is_best, val_m["f1"] if val_m else None)

                # Ghi CSV an toàn
                with open(self.history_csv_path, mode="a", newline="", encoding="utf-8") as f:
                    writer = csv.writer(f)
                    writer.writerow([
                        epoch, f"{train_m['loss']:.4f}", f"{train_m['acc']:.4f}", f"{train_m['recall']:.4f}", f"{train_m['f1']:.4f}",
                        f"{val_m['loss']:.4f}" if val_m else "", f"{val_m['acc']:.4f}" if val_m else "",
                        f"{val_m['recall']:.4f}" if val_m else "", f"{val_m['f1']:.4f}" if val_m else "",
                        f"{lr:.6e}", f"{elapsed:.2f}"
                    ])

                # Early Stopping
                if val_m and self.early_stopper and self.early_stopper.step(val_m["f1"], epoch):
                    print()
                    print(f"[*] Kích hoạt Early Stopping tại Epoch {epoch}! Kỷ nguyên tối ưu: Epoch {self.early_stopper.best_epoch}")
                    break

                if self.scheduler:
                    self.scheduler.step()
                if self.cfg.empty_cache_interval > 0 and epoch % self.cfg.empty_cache_interval == 0:
                    torch.cuda.empty_cache()

            print()
            print("[✓] QUÁ TRÌNH HUẤN LUYỆN ĐÃ HOÀN TẤT THÀNH CÔNG!")
        finally:
            if hasattr(self, "writer") and self.writer is not None:
                self.writer.flush()
                self.writer.close()
                print("[*] Đã đóng TensorBoard SummaryWriter an toàn.")


## 🚀 Section 7: Thực thi Huấn luyện Tương tác & Khôi phục Checkpoint
Khởi tạo đối tượng `KaggleTrainer` và lựa chọn kịch bản tiếp tục huấn luyện nếu cần.

In [ ]:
# 1. Khởi tạo và kiểm tra tham số mô hình
trainer = KaggleTrainer(cfg)

total_params = sum(p.numel() for p in trainer.model.parameters())
trainable_params = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
print(f"[*] Tổng số tham số ConvGRUClassifier: {total_params:,}")
print(f"[*] Số tham số huấn luyện (Trainable) : {trainable_params:,}")

In [ ]:
# ==============================================================================
# 2. CẤU HÌNH TIẾP TỤC HUẤN LUYỆN (RESUME CHECKPOINT SELECTION)
# ==============================================================================
# Nếu bạn muốn tiếp tục huấn luyện từ một checkpoint, hãy bỏ dấu comment '#'
# ở MỘT trong các tùy chọn dưới đây trước khi chạy cell tiếp theo:

# [Tùy chọn A] Tiếp tục từ checkpoint gần nhất (last.pt):
# trainer.load_checkpoint("last")

# [Tùy chọn B] Tiếp tục từ checkpoint có điểm F1 cao nhất (best.pt):
# trainer.load_checkpoint("best")

# [Tùy chọn C] Tiếp tục từ một epoch cụ thể đã lưu (ví dụ: Epoch 5 -> nạp epoch_5.pt):
# trainer.load_checkpoint(5)

# [Tùy chọn D] Tiếp tục từ tệp checkpoint đính kèm từ Kaggle Dataset (Input):
# trainer.load_checkpoint("/kaggle/input/my-driver-guardian-weights/epoch_10.pt")

print(f"[*] Trạng thái bắt đầu: Sẽ huấn luyện từ Epoch {trainer.start_epoch} đến Epoch {trainer.cfg.epochs}")

In [ ]:
# 3. Bắt đầu vòng lặp huấn luyện mô hình!
# Quá trình sẽ hiển thị thanh tiến trình trực quan tqdm và bảng thông số từng epoch.
trainer.train()

## 📊 Section 8: Phân tích & Trực quan hóa Đồ thị Học tập (Visualizations)
Trực quan hóa diễn biến hàm mất mát Loss, Độ chính xác Accuracy, Chỉ số F1-Score và Ma trận Nhầm lẫn Confusion Matrix.

In [ ]:
# 1. Vẽ đồ thị Loss, Accuracy, F1 và Learning Rate qua các epochs
import pandas as pd

if Path(cfg.history_csv_path).exists():
    df = pd.read_csv(cfg.history_csv_path)
    if len(df) > 0:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        # Đồ thị 1: Loss
        axes[0, 0].plot(df['epoch'], df['train_loss'], label='Train Loss', color='#1f77b4', marker='o', linewidth=2)
        if 'val_loss' in df and df['val_loss'].dropna().count() > 0:
            axes[0, 0].plot(df['epoch'], df['val_loss'], label='Val Loss', color='#ff7f0e', marker='s', linewidth=2)
        axes[0, 0].set_title('Cross-Entropy Loss', fontsize=13, fontweight='bold')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].grid(True, linestyle='--', alpha=0.6)
        axes[0, 0].legend()

        # Đồ thị 2: F1-Score
        axes[0, 1].plot(df['epoch'], df['train_f1'], label='Train F1', color='#2ca02c', marker='o', linewidth=2)
        if 'val_f1' in df and df['val_f1'].dropna().count() > 0:
            axes[0, 1].plot(df['epoch'], df['val_f1'], label='Val F1', color='#d62728', marker='s', linewidth=2)
        axes[0, 1].set_title('Macro F1-Score', fontsize=13, fontweight='bold')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('F1-Score')
        axes[0, 1].grid(True, linestyle='--', alpha=0.6)
        axes[0, 1].legend()

        # Đồ thị 3: Accuracy
        axes[1, 0].plot(df['epoch'], df['train_acc'], label='Train Acc', color='#9467bd', marker='o', linewidth=2)
        if 'val_acc' in df and df['val_acc'].dropna().count() > 0:
            axes[1, 0].plot(df['epoch'], df['val_acc'], label='Val Acc', color='#8c564b', marker='s', linewidth=2)
        axes[1, 0].set_title('Classification Accuracy', fontsize=13, fontweight='bold')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Accuracy')
        axes[1, 0].grid(True, linestyle='--', alpha=0.6)
        axes[1, 0].legend()

        # Đồ thị 4: Learning Rate
        axes[1, 1].plot(df['epoch'], df['lr'], label='Cosine LR', color='#e377c2', marker='^', linewidth=2)
        axes[1, 1].set_title('Learning Rate Schedule', fontsize=13, fontweight='bold')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Learning Rate')
        axes[1, 1].grid(True, linestyle='--', alpha=0.6)
        axes[1, 1].legend()

        plt.tight_layout()
        plot_path = Path(cfg.output_dir) / "training_curves.png"
        plot_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(plot_path, dpi=300)
        plt.show()
        print(f"[✓] Đã lưu biểu đồ học tập tại: {plot_path}")

In [ ]:
# 2. Đánh giá Ma trận Nhầm lẫn Confusion Matrix của mô hình tối ưu nhất (best.pt)
best_ckpt_path = Path(cfg.checkpoint_dir) / "best.pt"
if best_ckpt_path.exists() and len(trainer.val_loader.dataset) > 0:
    print(f"[*] Đang nạp trọng số tối ưu nhất từ: {best_ckpt_path}")
    best_ckpt = torch.load(best_ckpt_path, map_location=trainer.device, weights_only=False)
    trainer.model.load_state_dict(best_ckpt["model_state_dict"])
    
    val_metrics, all_preds, all_targets = trainer.validate_epoch(epoch=0)
    cm = confusion_matrix(all_targets, all_preds, labels=[0, 1])
    
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['0_Alert', '1_Drowsy'],
                yticklabels=['0_Alert', '1_Drowsy'],
                cbar=False, annot_kws={'size': 14})
    plt.title(f"Confusion Matrix (Best Model - Val F1: {val_metrics['f1']:.4f})", fontsize=12, fontweight='bold')
    plt.xlabel("Predicted Label", fontsize=11)
    plt.ylabel("Ground Truth Label", fontsize=11)
    
    cm_path = Path(cfg.output_dir) / "confusion_matrix_best.png"
    plt.savefig(cm_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"[✓] Đã lưu ma trận nhầm lẫn tại: {cm_path}")
    print(f"[✓] Best Validation Metrics: Acc={val_metrics['acc']:.4f}, Recall={val_metrics['recall']:.4f}, F1={val_metrics['f1']:.4f}")


## 📦 Section 9: Đóng gói Checkpoints & Tải về Máy tính (Export Artifacts)
Tự động nén toàn bộ thư mục `checkpoints/`, `logs/`, và hình ảnh kết quả thành một file `.zip` duy nhất trong `/kaggle/working/` để tải về thuận tiện.

In [ ]:
# Tự động nén các kết quả trọng tâm thành file zip để download từ Kaggle Output
zip_output_path = Path("/kaggle/working/driver_guardian_artifacts.zip" if IS_KAGGLE else "driver_guardian_artifacts.zip")

print("[*] Đang đóng gói các tệp checkpoint và logs trọng tâm...")
with zipfile.ZipFile(zip_output_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    # 1. Thêm checkpoint tốt nhất và checkpoint cuối cùng (tránh nén lặp lại epoch_*.pt gây tràn 20GB đĩa)
    for ckpt_name in ("best.pt", "last.pt"):
        cp = Path(cfg.checkpoint_dir) / ckpt_name
        if cp.exists():
            zipf.write(cp, arcname=f"checkpoints/{ckpt_name}")
    
    # 2. Thêm file CSV và biểu đồ hình ảnh
    if Path(cfg.history_csv_path).exists():
        zipf.write(cfg.history_csv_path, arcname=Path(cfg.history_csv_path).name)
    if Path(cfg.output_dir).exists():
        for p in Path(cfg.output_dir).rglob("*"):
            if p.is_file() and p.suffix in (".png", ".json", ".txt"):
                zipf.write(p, arcname=p.relative_to(Path(cfg.output_dir).parent))

    # 3. Thêm toàn bộ thư mục sự kiện TensorBoard logs (.tfevents)
    tb_dir = Path(cfg.tb_log_dir)
    if tb_dir.exists():
        for p in tb_dir.rglob("*"):
            if p.is_file():
                arc_name = Path("logs") / "tensorboard" / p.relative_to(tb_dir)
                zipf.write(p, arcname=str(arc_name))

file_size_mb = zip_output_path.stat().st_size / (1024 * 1024)
print(f"[✓] ĐÓNG GÓI THÀNH CÔNG: {zip_output_path} ({file_size_mb:.2f} MB)")
print("=" * 80)
print("🎉 HƯỚNG DẪN TẢI ARTIFACTS TRÊN KAGGLE:")
print("1. Nhìn sang cột phải của màn hình Kaggle Notebook, chọn mục 'Output'.")
print("2. Tìm tệp 'driver_guardian_artifacts.zip'.")
print("3. Nhấp vào biểu tượng ba chấm (...) bên cạnh tệp và chọn 'Download'.")
print("=" * 80)
